<a href="https://colab.research.google.com/github/MdRimon29/borno-lm/blob/main/src/pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the tokenizer dependency.
!pip install -q sentencepiece

In [2]:
# Install the Hugging Face datasets library.
!pip install -q datasets

In [3]:
# Mount Google Drive so checkpoints persist after the Colab session ends.
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# Import the libraries required for pretraining.
import math
import random
from pathlib import Path

import numpy as np
import sentencepiece as spm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [5]:
# Set random seeds and select the available training device.
def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

set_seed(123)

print("Device:", device)

Device: cuda


In [6]:
# Create the Borno-LM directories on Google Drive.
ROOT_DIR = Path("/content/drive/MyDrive/borno-lm")

DATA_DIR = ROOT_DIR / "data"
CHECKPOINT_DIR = ROOT_DIR / "checkpoints"
TOKENIZER_DIR = ROOT_DIR / "tokenizer"

for path in [DATA_DIR, CHECKPOINT_DIR, TOKENIZER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(ROOT_DIR)

/content/drive/MyDrive/borno-lm


In [8]:
# Load a small Bangla subset from the Hugging Face corpus.
from datasets import load_dataset

dataset = load_dataset(
    "ahmed-farhanur-rashid/bn-foundational-pretrain-corpus",
    "bangla_corpus",
    split="train",
    streaming=True,
)

Resolving data files:   0%|          | 0/55 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/55 [00:00<?, ?it/s]